# 📈 Notebook 2: The Phi (φ) Accrual Failure Detector

Notebook 1 left us with a dilemma: a single timeout is either too jumpy or too slow, and the "right" value depends on the network, the workload, even the time of day.

**Hayashibara et al. (2004)** proposed a radically different interface:

> Instead of returning *alive / dead*, output a continuous **suspicion value** `φ` that grows the longer we go without a heartbeat.

Each caller picks its own threshold: cache routers may act at `φ ≈ 3`; leader fencing may wait for `φ ≈ 12`. The detector itself makes **no decision** — it just gives you a number that means *"how surprised am I that I haven't heard from this node yet?"*

This is the algorithm used by **Apache Cassandra**, **Akka Cluster**, and **ScyllaDB**.


## 1. The math, one line at a time

Keep a sliding window of the last `N` inter-arrival times (e.g. the last 200 heartbeat gaps). Treat them as samples from a Normal distribution with mean `μ` and std-dev `σ` (Cassandra/Akka approximation — the original paper uses an exponential).

Let `Δ = t_now − t_last_heartbeat` be the current silence.

1. Compute the tail probability `P(interval ≥ Δ)` — *"how likely is it that a real interval would be at least this long?"*
2. Squeeze it to a more human scale: `φ = -log₁₀(P)`.

Quick feel for the numbers:

| `φ` | `P(interval ≥ Δ)` ≈ | Meaning |
|---:|---:|---|
| 1  | 10⁻¹ (10%)   | Slightly late — nothing to worry about |
| 3  | 10⁻³ (0.1%)  | Getting suspicious |
| 8  | 10⁻⁸         | **Cassandra's default — act now** |
| 12 | 10⁻¹²        | **Akka's default — absolute certainty** |

The survival function of a Normal distribution is `P(X ≥ Δ) = ½ · erfc((Δ−μ) / (σ·√2))`. Python ships `math.erfc` so we can compute it in one line.


## 2. Implementation

Tiny, dependency-free, commented so you can port it anywhere.


In [ ]:
import math
from collections import deque

class PhiAccrual:
    """Phi accrual failure detector (Normal approximation, the Cassandra flavor).

    Usage:
        d = PhiAccrual()
        d.heartbeat(t)    # whenever a heartbeat arrives
        d.phi(now)        # any time you want the current suspicion
    """

    def __init__(self, window=200, min_std=0.1, warmup=10):
        # sliding window of recent inter-arrival gaps
        self.intervals = deque(maxlen=window)
        self.last = None
        # floor on std-dev: on super-stable links the measured std can be tiny,
        # which makes phi explode on even a sliver of lateness. Cassandra uses 0.1s.
        self.min_std = min_std
        # need at least this many samples before we trust our statistics
        self.warmup = warmup

    def heartbeat(self, t: float) -> None:
        if self.last is not None:
            self.intervals.append(t - self.last)
        self.last = t

    def phi(self, now: float) -> float:
        # Not enough history yet -> no opinion (avoids cold-start false alarms)
        if self.last is None or len(self.intervals) < self.warmup:
            return 0.0
        mean = sum(self.intervals) / len(self.intervals)
        var  = sum((x - mean) ** 2 for x in self.intervals) / len(self.intervals)
        std  = max(math.sqrt(var), self.min_std)

        delta = now - self.last
        z     = (delta - mean) / std
        # P(interval >= delta) under Normal(mean, std)
        p     = 0.5 * math.erfc(z / math.sqrt(2))
        # clamp to avoid log10(0) when delta is enormous (keeps phi finite)
        p     = max(p, 1e-20)
        return -math.log10(p)

### First: is this the formula from the paper?

Hayashibara et al. define

$$\varphi(t_{now}) = -\log_{10}\bigl(P_{later}(t_{now} - T_{last})\bigr)$$

where `P_later(Δ)` is the probability that the next heartbeat arrives **more than** `Δ` after
the previous one — the survival function of the inter-arrival distribution. Everything hinges
on that being a survival function and not, say, a rescaled timeout, so let's check `phi()`
against normal-tail values computed independently.

In [ ]:
from statistics import NormalDist

def detector_with(mean, std, pairs=200):
    """A detector primed with a history whose mean and population std are exact.

    Alternating gaps of mean+std and mean-std, in equal numbers, give a sample mean
    of exactly `mean` and a population std of exactly `std` — so any difference from
    the reference below is a bug in phi(), not estimation noise.
    """
    d = PhiAccrual(window=2 * pairs, min_std=0.0, warmup=2)
    t = 0.0
    d.heartbeat(t)                       # first beat: starts the clock, no interval yet
    for _ in range(pairs):
        t += mean + std; d.heartbeat(t)
        t += mean - std; d.heartbeat(t)
    return d

MEAN, STD = 1.0, 0.25
d = detector_with(MEAN, STD)
assert len(d.intervals) == 400
assert abs(sum(d.intervals) / len(d.intervals) - MEAN) < 1e-9

print(f"{'Δ (silence)':>12} {'z':>4} {'phi (ours)':>11} {'-log10(normal tail)':>21}")
for z in (0, 1, 2, 3, 4, 6):
    delta = MEAN + z * STD
    ours = d.phi(d.last + delta)
    reference = -math.log10(1 - NormalDist(MEAN, STD).cdf(delta))   # independent
    print(f'{delta:>12.2f} {z:>4} {ours:>11.4f} {reference:>21.4f}')
    assert abs(ours - reference) < 1e-6, (z, ours, reference)

# Two landmarks worth knowing by heart:
#   Δ = mean       -> P = 0.5      -> phi = 0.301
#   Δ = mean + 3σ  -> P = 0.00135  -> phi = 2.87
assert abs(d.phi(d.last + MEAN) - 0.30103) < 1e-4
assert abs(d.phi(d.last + MEAN + 3 * STD) - 2.8697) < 1e-3
print("\n✔ phi matches -log10(P(interval > Δ)) exactly — the paper's formula,")
print("  not a timeout with a logarithm painted on it")

### Second: does it *accrue*?

The other half of the claim is in the name. A fixed timeout is a step function: 0, 0, 0, DEAD.
An accrual detector must produce a value that climbs smoothly and without bound as the silence
stretches, so that different callers can pick different thresholds and get *different*
detection times out of the same signal.

If `phi` were secretly a rebadged timeout, every threshold would fire at the same instant.

In [ ]:
d = detector_with(1.0, 0.25)
deltas = [i * 0.05 for i in range(1, 61)]        # 0.05s .. 3.0s of silence
series = [d.phi(d.last + x) for x in deltas]

# Strictly increasing with the length of the silence — never flat, never a step.
assert all(b > a for a, b in zip(series, series[1:])), 'phi is not strictly increasing'
assert series[0] < 0.01, series[0]
assert series[-1] > 12, series[-1]

# Note the ceiling: `phi()` clamps the tail probability at 1e-20, so phi saturates
# at 20 and stops rising. That is deliberate (it keeps the value finite) and harmless
# — every useful threshold is far below it — but it does mean phi is only strictly
# increasing up to that point. Worth knowing before you write phi > 25 somewhere.
assert d.phi(d.last + 10.0) == 20.0

# Distinct thresholds fire at distinct times. This is the property a timeout cannot
# have, and it is what lets one detector serve several callers at once.
def crossing(threshold):
    return next(x for x, ph in zip(deltas, series) if ph > threshold)

times = {th: crossing(th) for th in (1, 3, 8, 12)}
print(f"{'threshold':>10} {'silence before it fires':>26}")
for th, x in times.items():
    print(f'{th:>10} {x:>24.2f}s')
ordered = [times[th] for th in (1, 3, 8, 12)]
assert ordered == sorted(ordered) and len(set(ordered)) == 4, times
print('\n✔ four thresholds, four different firing times, one shared signal')
print('  (a fixed timeout would collapse all four into the same instant)')

## 3. Replay the same scenario from notebook 1

Same jittery heartbeat stream, same crash at `t = 20s`. This time we watch `φ` over time.

In [ ]:
import random, matplotlib.pyplot as plt

random.seed(7)
INTERVAL, JITTER, TOTAL, DEAD_AT = 1.0, 0.4, 30.0, 20.0

def make_trace(dead_at=DEAD_AT, total=TOTAL, blip=None):
    beats, now = [], 0.0
    while now < total:
        now += INTERVAL + random.uniform(-JITTER, JITTER)
        if now >= dead_at:
            break
        if blip is not None and blip[0] <= now < blip[0] + blip[1]:
            continue  # pretend the heartbeat was dropped by the network
        beats.append(now)
    return beats

heartbeats = make_trace()

def run_phi(beats, total=TOTAL, step=0.05):
    det = PhiAccrual()
    ts, phis = [], []
    i, t = 0, 0.0
    while t < total:
        while i < len(beats) and beats[i] <= t:
            det.heartbeat(beats[i]); i += 1
        ts.append(t); phis.append(det.phi(t))
        t += step
    return ts, phis

ts, phis = run_phi(heartbeats)

plt.figure(figsize=(10, 3.5))
plt.plot(ts, phis, label='φ')
plt.axhline(1,  color='gold',    linestyle=':', label='φ=1  (mild)')
plt.axhline(8,  color='tab:red', linestyle='--', label='φ=8  (Cassandra)')
plt.axhline(12, color='purple',  linestyle='-.', label='φ=12 (Akka)')
plt.axvline(DEAD_AT, color='black', linestyle=':', label='real crash')
plt.plot(heartbeats, [0.2]*len(heartbeats), '|', color='tab:green', label='heartbeat')
plt.xlabel('time (s)'); plt.ylabel('φ'); plt.legend(loc='upper left', fontsize=8)
plt.title('Phi stays near zero during healthy jitter and rockets up after the real crash')
plt.show()

crossings = {}
for th in (1, 8, 12):
    cross = next((t for t, p in zip(ts, phis) if p > th), None)
    crossings[th] = cross
    print(f'threshold φ={th:>2}: crossed at t={cross:.2f}s  ({cross - DEAD_AT:+.2f}s vs real crash)' if cross else f'threshold φ={th}: never crossed')

assert all(c is not None for c in crossings.values()), crossings
# Higher confidence must cost time, in order — that ordering is the whole interface.
assert crossings[1] <= crossings[8] <= crossings[12], crossings

healthy = [p for t, p in zip(ts, phis) if 5.0 < t < DEAD_AT]

# ⚠️ Read the φ=1 line carefully: it crosses BEFORE the crash. That is not a bug and
# not a false alarm — φ=1 means "P ≈ 10%", i.e. "this gap is a bit long", which on a
# ±0.4s-jitter link happens several times a minute. It is the tier you use for
# free, reversible reactions (shift some reads elsewhere), never for eviction.
assert crossings[1] < DEAD_AT, 'phi=1 is expected to fire on a healthy jittery link'

# The ACTION thresholds are the ones that must stay silent while the node is alive.
assert max(healthy) < 8, f'phi reached {max(healthy):.1f} on a healthy node'
assert crossings[8] > DEAD_AT and crossings[12] > DEAD_AT, crossings
assert phis[-1] > 12

print(f'\n✔ max phi while alive was {max(healthy):.2f} — below both action thresholds')
print(f'  φ=1 fired at t={crossings[1]:.1f}s (alive, and that is fine: it is advisory)')
print(f'  φ=8 fired {crossings[8]-DEAD_AT:.1f}s after the crash, φ=12 at {crossings[12]-DEAD_AT:.1f}s')

## 4. A network blip — and an uncomfortable result

Real networks lose bursts of packets. Let's drop **every** heartbeat for 1.5 seconds in the
middle of the node's life. The node is perfectly alive the whole time.

A fixed 1.0-second timeout obviously screams FAILURE here. The tempting story is that phi
quietly rides it out. Run it and read the number before believing that.


In [ ]:
random.seed(7)
blip_trace = make_trace(dead_at=TOTAL, blip=(15.0, 1.5))  # node never actually dies

ts_b, phis_b = run_phi(blip_trace)

peak = max(p for t, p in zip(ts_b, phis_b) if 15.0 <= t <= 17.0)
recovered = max(p for t, p in zip(ts_b, phis_b) if 18.0 <= t <= 25.0)
print(f'peak φ during 1.5s blip: {peak:.2f}')
print(f'φ once beats resume    : {recovered:.2f}')
print(f'would a Cassandra detector (φ>8) false-alarm? {"YES" if peak > 8 else "NO"}')
print(f'would a fixed 1.0s timeout false-alarm?      YES (blip is 1.5s long)')

# Suspicion rises during the blip and collapses afterwards — both required.
# But it does NOT stay under 8: on this link a Cassandra-default detector evicts a
# perfectly healthy node. Assert the uncomfortable result rather than hide it.
assert peak > 1.0, f'phi barely moved during the blip: {peak:.2f}'
assert peak > 8, f'expected a false positive at the default threshold, got {peak:.2f}'
assert recovered < 1.0, f'phi did not recover after the blip: {recovered:.2f}'
print(f'\n💥 φ peaked at {peak:.2f}. At the Cassandra default (φ>8) this healthy node')
print('   gets evicted. Phi is not magic — it moved the knob, it did not remove it.')

plt.figure(figsize=(10, 3.5))
plt.plot(ts_b, phis_b, label='φ')
plt.axvspan(15.0, 16.5, color='grey', alpha=0.25, label='network blip')
plt.axhline(8, color='tab:red', linestyle='--', label='φ=8')
plt.xlabel('time (s)'); plt.ylabel('φ'); plt.legend()
plt.title('Phi rises sharply during the blip and recovers — but it does cross 8')
plt.show()

### Why phi was right and the threshold was wrong

Phi is not malfunctioning — it is reporting exactly what it was asked. The learned interval
distribution here is `μ ≈ 1.0s, σ ≈ 0.23s`. A 2.5-second silence is a **6-sigma** event under
that distribution, and `-log₁₀` of a 6-sigma tail really is around 9. Phi's answer of "this is
astronomically unlikely" is correct; it is merely wrong about the *cause*, because the Normal
model has never seen a burst loss and has no way to represent one.

So the lesson is not "use phi instead of tuning". It is:

- **The threshold encodes the blip you intend to survive.** To ride out 1.5s of packet loss on a
  1s cadence, φ=8 is too low. Cassandra's default assumes a LAN.
- **`min_std` is the other half of the knob.** Floor the standard deviation at something that
  reflects the *worst* the link does rather than its average, and the same blip stops looking
  like a 6-sigma event.

Let's tune it and confirm both properties: the blip survives, and the real crash is still
caught quickly.

In [ ]:
def peak_and_detect(min_std, threshold):
    """Peak phi during the blip (node alive), and detection delay on the real crash."""
    det = PhiAccrual(min_std=min_std)
    ts_x, phis_x = [], []
    i, t = 0, 0.0
    while t < TOTAL:
        while i < len(blip_trace) and blip_trace[i] <= t:
            det.heartbeat(blip_trace[i]); i += 1
        ts_x.append(t); phis_x.append(det.phi(t)); t += 0.05
    blip_peak = max(p for tt, p in zip(ts_x, phis_x) if 15.0 <= tt <= 18.0)

    det = PhiAccrual(min_std=min_std)          # same settings, the crash trace
    i, t, fired = 0, 0.0, None
    while t < TOTAL:
        while i < len(heartbeats) and heartbeats[i] <= t:
            det.heartbeat(heartbeats[i]); i += 1
        if fired is None and t >= 1.0 and det.phi(t) > threshold:
            fired = t
        t += 0.05
    return blip_peak, (fired - DEAD_AT if fired is not None and fired >= DEAD_AT else None)

print(f"{'min_std':>8} {'threshold':>10} {'peak φ in blip':>15} {'false alarm?':>13} {'crash detected':>15}")
for min_std, threshold in [(0.1, 8), (0.1, 16), (0.6, 8), (0.6, 12)]:
    peak_x, delay = peak_and_detect(min_std, threshold)
    fa = 'YES' if peak_x > threshold else 'no'
    shown = f'{delay:+.1f}s' if delay is not None else 'never'
    print(f'{min_std:>8.1f} {threshold:>10} {peak_x:>15.2f} {fa:>13} {shown:>15}')

# Default LAN settings on this WAN-ish link: false alarm.
assert peak_and_detect(0.1, 8)[0] > 8

# Floor the std to match what the link really does and the blip stops being a
# 6-sigma event — while the actual crash is still detected within a few seconds.
peak_tuned, delay_tuned = peak_and_detect(0.6, 8)
assert peak_tuned < 8, peak_tuned
assert delay_tuned is not None and delay_tuned < 5.0, delay_tuned
print(f'\n✔ with min_std=0.6 the blip peaks at {peak_tuned:.2f} (no action taken) and the')
print(f'  real crash is still caught {delay_tuned:.1f}s after it happens')

### So what *is* the advantage over a fixed timeout?

Fair question, having just spent a cell tuning a threshold. Two things, and neither of them is
"no tuning required":

1. **One signal, many decisions.** A timeout produces a boolean, so every caller is stuck with
   it. Phi produces a number, so a cache can react at φ=3 while leader fencing waits for φ=12 —
   from the *same* heartbeat stream, with no extra probes. Notebook 3 builds exactly that.
2. **It adapts to the cadence it observes.** This is the part a fixed threshold genuinely
   cannot do: a node that is simply *slower* than its peers is indistinguishable from a dead one
   to a timeout, while phi learns its rhythm and stops complaining.

The second is checkable, so let's check it.

In [ ]:
# A node that is alive and well but beats every 1.5s instead of every 1.0s —
# a smaller instance, a busier host, one extra network hop.
random.seed(11)
slow_beats, t = [], 0.0
while t < TOTAL:
    t += 1.5 + random.uniform(-0.3, 0.3)
    slow_beats.append(t)

def fixed_timeout_down_fraction(beats, timeout, total=TOTAL, step=0.05):
    last_seen, i, t, down, samples = None, 0, 0.0, 0, 0
    while t < total:
        while i < len(beats) and beats[i] <= t:
            last_seen = beats[i]; i += 1
        if last_seen is not None:
            samples += 1
            if t - last_seen > timeout:
                down += 1
        t += step
    return down / samples

TO = 1.2                       # a timeout tuned for the 1.0s cadence
down_fraction = fixed_timeout_down_fraction(slow_beats, TO)

# Phi, on the same trace, learns that 1.5s IS this node's normal.
det = PhiAccrual()
peaks, i, t = [], 0, 0.0
while t < TOTAL:
    while i < len(slow_beats) and slow_beats[i] <= t:
        det.heartbeat(slow_beats[i]); i += 1
    if t > 15.0:                                  # after warm-up
        peaks.append(det.phi(t))
    t += 0.05

print(f'fixed timeout {TO}s : flags DOWN {down_fraction:.0%} of the time (the node is fine)')
print(f'phi accrual        : max φ = {max(peaks):.2f} (no action threshold crossed)')

assert down_fraction > 0.2, 'the fixed timeout should be flapping badly here'
assert max(peaks) < 8, f'phi should have adapted, got {max(peaks):.2f}'
print('\n✔ this is the thing a fixed threshold cannot do: the node is not late,')
print('  it just has a different rhythm, and phi learned that from the data')

## 5. Side-by-side vs fixed timeout

One picture, same trace with the blip:

In [ ]:
def fixed_timeout_flags(beats, timeout, total=TOTAL, step=0.05):
    times, flags = [], []
    last_seen = 0.0; i = 0; t = 0.0
    while t < total:
        while i < len(beats) and beats[i] <= t:
            last_seen = beats[i]; i += 1
        times.append(t)
        flags.append(1 if (t - last_seen) > timeout else 0)
        t += step
    return times, flags

t_ft, f_ft = fixed_timeout_flags(blip_trace, timeout=1.0)

fig, (a, b) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
a.fill_between(t_ft, 0, f_ft, step='pre', color='tab:red', alpha=0.4)
a.axvspan(15.0, 16.5, color='grey', alpha=0.25)
a.set_title('Fixed 1.0s timeout: false DOWN for the entire blip')
a.set_yticks([0,1]); a.set_ylabel('DOWN?')

b.plot(ts_b, phis_b)
b.axhline(8, color='tab:red', linestyle='--')
b.axvspan(15.0, 16.5, color='grey', alpha=0.25)
b.set_title(f'Phi accrual, default min_std: peaks at {max(phis_b):.1f} — also a false alarm here')
b.set_xlabel('time (s)'); b.set_ylabel('φ')
plt.tight_layout(); plt.show()

# Same trace, same living node: at DEFAULT settings BOTH detectors get it wrong.
# The honest comparison is not "timeout bad, phi good" — it is that phi's mistake is
# tunable per-caller in units of probability, while the timeout's is not.
false_down_seconds = sum(f_ft) * 0.05
assert false_down_seconds > 0.5, 'the fixed timeout should have false-alarmed here'
assert max(phis_b) > 8, 'and so does phi at min_std=0.1 — see the tuning cell above'
print(f'fixed 1.0s timeout       : {false_down_seconds:.2f}s falsely marked DOWN')
print(f'phi accrual (min_std=0.1): peak {max(phis_b):.2f} — crosses 8 as well')
print(f'phi accrual (min_std=0.6): peak {peak_tuned:.2f} — no false alarm, crash still caught')

## 6. Why phi wins — summary

|                       | Fixed timeout             | Phi accrual                                   |
|-----------------------|---------------------------|-----------------------------------------------|
| Adapts to network     | ❌ you hand-tune           | ✅ learns μ and σ from the window             |
| One value fits all    | ❌ every app shares it     | ✅ each caller picks its threshold            |
| False positives       | many under jitter         | rare; suspicion grows smoothly                |
| Crash detection speed | fixed by the timeout      | fast when network is calm, patient when noisy |

**Practical tips**

- `window ≈ 100–1000` recent intervals is plenty.
- Floor the std-dev (`min_std ≈ 0.1s`). Without it, an unnaturally stable link makes `φ` explode on the tiniest lateness.
- Don't mix short-lived bursts (like gossip pings) with the steady-state window. Reset the detector on reconnect.
- Phi itself is **passive**. What to *do* when it crosses your threshold (back off traffic, start a replacement, trigger leader election) is the topic of notebook 3.

👉 Continue with `03_real_world_cluster.ipynb`.